# 💰 Finance Advisor — *No Memory* Edition

A personal finance advisor built with the **OpenAI Agents SDK**.

The agent **tracks spending by category** and **gives budget advice**.

> ⚠️ **No memory / no session.** Every query is handled independently — the agent does **not** remember anything from earlier queries. This is intentional: the notebook demonstrates the *stateless* baseline. Watch **Test Query 2**, which asks about "everything I told you so far" — with no memory, the agent has no access to the spending logged in Query 1.

**Model:** `gpt-5-mini` — verified as a currently available OpenAI API model (July 2026).

## 1. Install dependencies

In [ ]:
%pip install -q openai-agents python-dotenv

## 2. Environment setup (dotenv)

Loads `OPENAI_API_KEY` from a local `.env` file. Create a `.env` next to this notebook containing:

```
OPENAI_API_KEY=sk-...
```

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from a local .env file (does not overwrite existing ones)
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OPENAI_API_KEY not found. Add it to a .env file next to this notebook."
print("✅ OPENAI_API_KEY loaded.")

## 3. Define the Finance Advisor agent

In [ ]:
from agents import Agent, Runner

INSTRUCTIONS = """
You are "Finance Advisor", a friendly and practical personal finance assistant.

For each user message:
1. Identify every expense the user mentions.
2. Categorize each expense (Groceries, Transport, Dining, Entertainment, Bills,
   Shopping, Health, Other).
3. Give a short per-category breakdown and the total, all in USD.
4. If the user shares a budget, compare their spending against it and suggest
   2-4 specific, realistic places to cut back.

Important:
- You have NO memory of previous messages. Use ONLY the information contained in
  the current message. If the user refers to things they "told you before" that
  are not present in this message, say clearly that you do not have that context
  and ask them to restate the numbers.
- Keep it concise and actionable. You are not a licensed financial advisor, so
  keep advice general and practical.
""".strip()

finance_advisor = Agent(
    name="Finance Advisor",
    instructions=INSTRUCTIONS,
    model="gpt-5-mini",
)

print(f"Agent '{finance_advisor.name}' ready on model '{finance_advisor.model}'.")

## 4. Test Query 1 — log this week's spending

Each query is a **fresh, independent run** (`Runner.run`) with no shared state.

In [ ]:
query_1 = "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week."

result_1 = await Runner.run(finance_advisor, query_1)
print(result_1.final_output)

## 5. Test Query 2 — ask for advice against a budget

This is a **separate, independent run**. Because there is no memory, the agent does *not* see the spending from Query 1 — it only receives the text below. Expect it to note that it lacks the earlier context.

In [ ]:
query_2 = (
    "My weekly budget is $250. Based on everything I told you so far, "
    "where can I cut spending next week?"
)

result_2 = await Runner.run(finance_advisor, query_2)
print(result_2.final_output)

## Why the two answers differ

Query 2 references "everything I told you so far," but this agent is **stateless** — each `Runner.run(...)` call starts with a blank slate. The spending from Query 1 is never carried over, so the advisor can only work with the `$250` budget contained in Query 2's own text.

To make the agent remember across queries, you would add **memory / sessions** (pass prior turns as conversation history, or use the SDK's session support) — that is the counterpart *with-memory* notebook.